# 실습 10: 드문 쪽을 놓치지 않게 만들기
- 상황: 어제 모델은 실제 불량 스물한 건 중 두 건만 잡았다
- 목표: 놓친 쪽을 줄이는 방법을 적용하고, 무엇을 내줬는지 함께 적는다

## Step 0. 어제 상태까지 재현하기

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

sensor_cols = df.columns.drop("result")
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

df["불량여부"] = (df["result"] == "불량").astype(int)

X = df[sensor_cols]
y = df["불량여부"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# class_weight 등 불균형 보정 없이 기본 설정 그대로 사용한다 (어제 상태 그대로)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

예측 = model.predict(X_test_scaled)

정확도 = (예측 == y_test).mean() * 100
불량예측건수 = int((예측 == 1).sum())
진짜불량건수 = int(((예측 == 1) & (y_test == 1)).sum())

print("학습용:", X_train.shape[0], "건, 불량", int(y_train.sum()), "건")
print("시험용:", X_test.shape[0], "건, 불량", int(y_test.sum()), "건")
print("정확도:", round(정확도, 2), "%")
print("불량이라 예측한 건수:", 불량예측건수, "건, 그중 진짜 불량", 진짜불량건수, "건")

학습용: 1253 건, 불량 83 건
시험용: 314 건, 불량 21 건
정확도: 92.99 %
불량이라 예측한 건수: 5 건, 그중 진짜 불량 2 건


day9, step_0과 결과 동일. (로지스틱 회귀 적용)
전/후 차이 비교 가능

---

## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 드문 쪽을 다루는 말

| 말 | 뜻 |
|---|---|
| 클래스 불균형 | 한쪽이 지나치게 드문 상태. 우리 데이터는 불량이 약 6.6%뿐이다 |
| 클래스 가중치 | 드문 쪽 한 건을 여러 건만큼 무겁게 세도록 모델에 알려주는 설정 |
| 언더샘플링 | 많은 쪽을 줄여서 양쪽 수를 맞추는 방법. 데이터를 버리게 된다 |
| 오버샘플링 | 드문 쪽을 늘려서 양쪽 수를 맞추는 방법. 없던 기록을 만들어 넣게 된다 |
| 재현율 | 실제 불량 중 몇 %를 잡았나. 오늘 올리려는 숫자 |
| 정밀도 | 불량이라 한 것 중 몇 %가 진짜였나. 오늘 내주게 될 숫자 |

---

## Step 2. 무게를 다르게 주기

In [11]:
# 표준화와 모델을 한 줄로 묶어주는 도구들을 불러온다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# class_weight="balanced" - 드문 쪽 한 건을 그만큼 무겁게 세라는 뜻. 오늘 추가한 것은 이 한 조각뿐이다
가중치모델 = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

# 학습용으로만 학습시킨다. 시험용은 여전히 건드리지 않는다
가중치모델.fit(X_train, y_train)

# 시험용 입력만 넣어 답을 받는다
가중치예측 = 가중치모델.predict(X_test)

print("불량이라고 예측한 건수:", 가중치예측.sum())
print("그중 진짜 불량:", ((가중치예측 == 1) & (y_test == 1)).sum())

불량이라고 예측한 건수: 75
그중 진짜 불량: 9


### 문법 노트 - 오늘 추가한 한 조각

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| class_weight="balanced" | 적은 쪽 한 건을 더 무겁게 세게 한다 | 그냥 두면 모델이 많은 쪽만 맞히고 만족해버린다 |
| max_iter=1000 | 답을 찾을 때까지 계산을 더 오래 하게 둔다 | 기본값으로는 다 못 찾고 멈췄다는 경고가 뜬다 |

---
## Step 3. 전후 숫자 비교하기

In [12]:
# 네 칸 표와 지표들을 계산해주는 도구를 불러온다
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

# 어제 예측과 오늘 예측을 나란히 놓고 같은 자로 잰다
for 이름, 예측 in [("손 안 댐", 예측), ("가중치", 가중치예측)]:
    # ravel - 네 칸짜리 표를 한 줄로 펴서 이름을 하나씩 붙여 받는다
    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()
    print(f"[{이름}]")
    print("  정확도:", round((예측 == y_test).mean() * 100, 2), "%")
    print("  잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
    print("  재현율:", round(recall_score(y_test, 예측), 3),
          "정밀도:", round(precision_score(y_test, 예측, zero_division=0), 3),
          "F1:", round(f1_score(y_test, 예측), 3))

[손 안 댐]
  정확도: 92.99 %
  잡은 불량: 2 / 놓친 불량: 19 / 헛경보: 3
  재현율: 0.095 정밀도: 0.4 F1: 0.154
[가중치]
  정확도: 75.16 %
  잡은 불량: 9 / 놓친 불량: 12 / 헛경보: 66
  재현율: 0.429 정밀도: 0.12 F1: 0.188


---
## Step 4. 많은 쪽을 줄여서 해보기

In [13]:
# 불량 개수만큼 양품을 무작위로 뽑아서, 학습용 안에서 양쪽 수를 맞춘다
불량_수 = int(y_train.sum())
양품_줄임_인덱스 = y_train[y_train == 0].sample(n=불량_수, random_state=42).index
불량_인덱스 = y_train[y_train == 1].index

줄인_인덱스 = 양품_줄임_인덱스.union(불량_인덱스)
X_train_줄임 = X_train.loc[줄인_인덱스]
y_train_줄임 = y_train.loc[줄인_인덱스]

# 시험용(X_test, y_test)은 그대로 둔다 - 줄이거나 늘리지 않는다
언더샘플모델 = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
언더샘플모델.fit(X_train_줄임, y_train_줄임)

언더샘플예측 = 언더샘플모델.predict(X_test)

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 언더샘플예측).ravel()

print("[줄인 학습용]")
print("  전체:", len(y_train_줄임), "건 (불량", int((y_train_줄임 == 1).sum()), "건 / 양품", int((y_train_줄임 == 0).sum()), "건)")
print("  줄이기 전 학습용:", len(y_train), "건 (불량", int((y_train == 1).sum()), "건 / 양품", int((y_train == 0).sum()), "건)")
print()
print("[시험용은 그대로인지 확인]")
print("  시험용:", len(y_test), "건 (불량", int(y_test.sum()), "건) - 줄이기 전과 같아야 한다")
print()
print("[줄인 학습용으로 만든 모델의 성적]")
print("  정확도:", round((언더샘플예측 == y_test).mean() * 100, 2), "%")
print("  잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
print("  재현율:", round(recall_score(y_test, 언더샘플예측), 3),
      "정밀도:", round(precision_score(y_test, 언더샘플예측, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 언더샘플예측), 3))

[줄인 학습용]
  전체: 166 건 (불량 83 건 / 양품 83 건)
  줄이기 전 학습용: 1253 건 (불량 83 건 / 양품 1170 건)

[시험용은 그대로인지 확인]
  시험용: 314 건 (불량 21 건) - 줄이기 전과 같아야 한다

[줄인 학습용으로 만든 모델의 성적]
  정확도: 71.02 %
  잡은 불량: 13 / 놓친 불량: 8 / 헛경보: 83
  재현율: 0.619 정밀도: 0.135 F1: 0.222


## Step 5. 전후 비교표

| 처리 | 정확도 | 잡은 불량 | 놓친 불량 | 헛경보 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|---|---|
| 손 안 댐 | [92.99]% | [2] | [19] | [3] | [0.095] | [0.4] | [0.154] |
| 가중치 주기 | [75.16]% | [9] | [12] | [66] | [0.429] | [0.12] | [0.188] |
| 많은 쪽 줄이기 | [71.02]% | [13] | [8] | [83] | [0.619] | [0.135] | [0.222] |

---
## Step 6. 정직한 처리의 선

- 세 방법 모두 **학습용에만** 적용했다
- 시험용 314건은 처음 나눈 그대로 두었다 (불량 21건 그대로)
- 시험용을 손보면 점수는 올라가지만, 현장에 나가는 순간 그 점수는 없다

---
## 직접 해보기 (도전) - 시험지까지 손대면 어떻게 되나

- 상황: 학습용에만 손대라고 했는데, 시험용에도 손대면 점수가 어떻게 나올까
- 할 일: 시험용을 반반으로 맞춰놓고 같은 모델의 점수를 다시 잰다
- 결과물: 두 줄짜리 비교표 1개

In [14]:
# 시험용에서 불량 개수만큼 양품을 무작위로 뽑아 반반짜리 시험용을 새로 만든다
# 원래 X_test, y_test는 건드리지 않고 새 이름으로 만든다
불량_인덱스_시험 = y_test[y_test == 1].index
양품_줄임_인덱스_시험 = y_test[y_test == 0].sample(n=len(불량_인덱스_시험), random_state=42).index

반반_인덱스 = 불량_인덱스_시험.union(양품_줄임_인덱스_시험)
X_test_반반 = X_test.loc[반반_인덱스]
y_test_반반 = y_test.loc[반반_인덱스]

# 가중치모델은 Step 2에서 학습시킨 것을 그대로 쓴다 - 다시 학습시키지 않는다
가중치예측_반반 = 가중치모델.predict(X_test_반반)

행들 = []
for 이름, y부분, 예측값 in [
    ("원래 시험용", y_test, 가중치예측),
    ("반반 시험용", y_test_반반, 가중치예측_반반),
]:
    행들.append({
        "시험용": 이름,
        "건수": len(y부분),
        "정확도(%)": round((예측값 == y부분).mean() * 100, 2),
        "재현율": round(recall_score(y부분, 예측값), 3),
        "정밀도": round(precision_score(y부분, 예측값, zero_division=0), 3),
        "F1": round(f1_score(y부분, 예측값), 3),
    })

비교표_반반 = pd.DataFrame(행들).set_index("시험용")
비교표_반반

,건수,정확도(%),재현율,정밀도,F1
시험용,,,,,
원래 시험용,314,75.16,0.429,0.12,0.188
반반 시험용,42,69.05,0.429,0.90,0.581


### 시험지를 손대면

| 채점 방식 | 건수 | 정확도 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|
| 원래 시험용 | [314] | [75.16]% | [0.429] | [0.12] | [0.188] |
| 반반으로 맞춘 시험용 | [42] | [69.05]% | [0.429] | [0.90] | [0.581] |